# Exploratory Data Analysis for Crop and Weather Data

## [Contents](#contents)

- [1. Overview](#overview)
- [2. Environment Configuration](#2-environment-configuration)
- [3. Crops](#3-crops)
- [4. Weather](#4-weather)
- [5. Combined Weather 7 Crop Data Analysis](#5-combined-weather--crop-data-analysis)

## [2. Environment Configuration](#contents)

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import numpy as np

In [ ]:
project_db  = 'data/project_db.db'

## [3. Crops](#contents)

The UK has a varied landscape with varying soil, weather, and topographic conditions. These could all be factors in crop yields. To reduce the influence of these, yield data for just East Anglia will be analysed.

### [3.1 UK Yearly Crop Yields](#contents)

- What do Crop yields look like over the years?
- Is there variation that might be explained by weather variation across the years?
- Which crop shows most promise for analysis?

#### UK Yearly Crop Data

In [ ]:
qry_yields = """
	SELECT 
		o.year,
		o."United Kingdom" AS UK_Oats,
		osr."United Kingdom" AS UK_OSR,
		sb."United Kingdom" AS UK_Spring_Barley,
		w."United Kingdom" AS UK_Wheat,
		wb."United Kingdom" AS UK_Winter_Barley

	FROM
		yields_oats o

		JOIN yields_OSR osr
			ON o.year = osr.year

		JOIN yields_spring_barley sb
			ON o.year = sb.year

		JOIN yields_wheat w
			ON o.year = w.year

		JOIN yields_winter_barley wb
			ON o.year = wb.year

	WHERE
		o.year >= 1998 AND o.year < 2026
"""

conn = sqlite3.connect(project_db)

df_yields = pd.read_sql(qry_yields, conn)

conn.close()

df_yields.head(10)

#### UK Yearly Crop Yields over Time Line Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
df.loc[:, 'year'] = pd.to_datetime(df['Year'], format='%Y').dt.year
plt.plot(df_yields.Year, df_yields.UK_Oats , color='tab:red', label='Oats')
plt.plot(df_yields.Year, df_yields.UK_OSR , color='tab:blue', label='OSR')
plt.plot(df_yields.Year, df_yields.UK_Spring_Barley , color='tab:green', label='Spring Barley')
plt.plot(df_yields.Year, df_yields.UK_Wheat , color='tab:purple', label='Wheat')
plt.plot(df_yields.Year, df_yields.UK_Winter_Barley , color='tab:orange', label='Winter Barley')


# add figure elements
plt.title(f'UK Annual Crop Yields (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Rainfall (mm)')
plt.legend(loc=2, ncols = 1)
plt.show()

Observations:

- Yields evidently vary from year to year.
- These variations appear to be consistent across crops.
- Wheat is the highest yielding crop.

#### UK Yearly Crop Yield Correlation Heatmap

A correlation Heatmap will show if the yields are correlated.

In [ ]:
# Drop Year column
df_yields_corr = df_yields.drop(columns=['Year'], inplace=False)
df_yields_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_yields_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- Wheat has high correlations with Winter Barley, Spring Barley and Oats.
- Oil Seed Rape (OSR) yield has little correlation with any other crop.

This suggests some underlying cause for at least Wheat, Spring Barley and Oats yields to fluctuate together. This could well be weather related, or ALSO might be related to growing location.

### [3.2 UK Country Wheat Crop Yields](#contents)

- For a single crop, how does wheat yield differ across the UK Countries?
- Does yield across countries correlate, well, or do they show variation?

##### UK Countries Wheat Data

In [ ]:
qry_regional_wheat_yields = """
	SELECT 
		Year,
		England,
		Scotland,
		Wales,
		"Northern Ireland" AS NI

	FROM
		yields_wheat

	WHERE
		year >= 1998 AND year < 2026
"""

conn = sqlite3.connect(project_db)

df_wheat_yields = pd.read_sql(qry_regional_wheat_yields, conn)

conn.close()

df_wheat_yields.head(10)

##### UK Counties Wheat Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

df = df_wheat_yields

# Create Month number field for x-axis and plot each year as a line on the same figure.
df.loc[:, 'Year'] = pd.to_datetime(df['Year'], format='%Y').dt.year
plt.plot(df.Year, df.England , color='tab:orange', label='England')
plt.plot(df.Year, df.Scotland , color='tab:blue', label='Scotland')
plt.plot(df.Year, df.Wales , color='tab:red', label='Wales')
plt.plot(df.Year, df.NI , color='tab:green', label='Northern Ireland')


# add figure elements
plt.title(f'UK Annual Wheat Yields Across Regions (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Wheat Yield')
plt.legend(loc=2, ncols = 1)
plt.show()

##### UK Counties Wheat Yield Correlation Heatmap

In [ ]:
# Drop Year column
df_wheat_yields_corr = df_wheat_yields.drop(columns=['Year'], inplace=False)
df_wheat_yields_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_wheat_yields_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- Correlation in wheat yield between the UK counties is not very high, with the best being between Wales and England.
- This indicates likely differences between these countries that may be related to weather or other factors such as soil quality, farming practices or topology.
- For an assessment of weather impact on crop yields, this needs to be minimised, with areas that have as consistent variation as possible.

### [3.3 England Region Wheat Crop Yields](#contents)

- For a single crop, how does the wheat yield differ across England Regions?
- Does yield across England regions correlate?

##### Regional England Wheat Yield Data

In [ ]:
qry_england_wheat_yields = """
	SELECT 
		Year,
		"North East" AS NE,
		"North West and Merseyside" AS NW,
		"Yorkshire & The Humber" AS Y,
		"East Midlands" AS EM,
		"West Midlands" AS WM,
		"Eastern" AS E,
		"South East and London" AS SE,
		"South West" AS SW

	FROM
		yields_wheat

	WHERE
		year >= 1998 AND year < 2026
"""

conn = sqlite3.connect(project_db)

df_wheat_yields_england = pd.read_sql(qry_england_wheat_yields, conn)

conn.close()

df_wheat_yields_england.head(10)

##### Regional England Wheat Yield Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

df = df_wheat_yields_england

# Create Month number field for x-axis and plot each year as a line on the same figure.
df.loc[:, 'Year'] = pd.to_datetime(df['Year'], format='%Y').dt.year
plt.plot(df.Year, df.NE , color='tab:blue', label='North East')
plt.plot(df.Year, df.NW , color='tab:orange', label='North West & Merseyside')
plt.plot(df.Year, df.Y , color='tab:green', label='Yorkshire & The Humber')
plt.plot(df.Year, df.EM , color='tab:red', label='East Midlands')
plt.plot(df.Year, df.WM , color='tab:purple', label='West Midlands')
plt.plot(df.Year, df.E , color='tab:brown', label='Eastern')
plt.plot(df.Year, df.SE , color='tab:pink', label='South East & London')
plt.plot(df.Year, df.SW , color='tab:grey', label='South West')


# add figure elements
plt.title(f'Regional England Wheat Yields (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Wheat Yield')
plt.legend(loc=2, ncols = 3)
plt.show()

##### Regional England Wheat Yield Correlation Heatmap

In [ ]:
# Drop Year column
df_wheat_yields_england_corr = df_wheat_yields_england.drop(columns=['Year'], inplace=False)
df_wheat_yields_england_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_wheat_yields_england_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- The North West has the lowest correlation with all other regions. This is apparent when looking at yield over the years from 1999 to 2025, much lower than all other regions.
- The highest correlations are in adjacent regions, such as South West and South East & London, West and East Midlands, or even South East & London with West Midlands and Yorkshire & The Humber.
- There are clearly regional influences in play. To reduce this noise and to identify the influence of weather factors, selecting a single region to focus on would be beneficial.

## [4. Weather](#contents)

- Focus on Midlands weather region as likely closest approximation for West Midlands Crop region.
- Correlation Heat Map for Annual Weather against Wheat Yield.
- Select most correlated, Correlation Heatmap each month against yield.

### [Rainfall](#contents)

#### 4.1 Rainfall Across Years

##### Get Data from database

In [ ]:
qry_uk_rf = """
	SELECT 
		year,
		jan,
		feb,
		mar,
		apr,
		may,
		jun,
		jul,
		aug,
		sep,
		oct,
		nov,
		dec

	FROM
		Rainfall
	WHERE
		year >= 1998 AND year < 2026
		AND 
		area = 'UK'
"""

conn = sqlite3.connect(project_db)

df_uk_rf = pd.read_sql(qry_uk_rf, conn)

conn.close()

df_uk_rf.head(10)

##### Pivot from Long to Short format dataframe for plotting

In [ ]:
# Un-pivot the data into long format.
df_uk_rf_long = df_uk_rf.melt(id_vars='year', var_name='month', value_name='rainfall')

# Create new field for date with month and year.
df_uk_rf_long['date'] = pd.to_datetime(df_uk_rf_long['year'].astype(str) + '-' + df_uk_rf_long['month'], format='%Y-%b')

# Sort data by the date and reset the index.
df_uk_rf_long = df_uk_rf_long.sort_values('date').reset_index(drop=True)

# Verify with sample.
df_uk_rf_long.head(14)

##### Timeseries Plot Rainfall between 1999 and 2025

In [ ]:
def plot_df(df, x, y, title="", xlabel='Date', ylabel='Value', dpi=100):
    plt.figure(figsize=(16,5), dpi=dpi)
    plt.plot(x, y, color='tab:red')
    plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.show()

plot_df(df_uk_rf_long, x=df_uk_rf_long.date, y=df_uk_rf_long.rainfall, title='UK Rainfall')

This figure shows some significant fluctuation. A close look at a few single years might show a pattern.

#### 4.2 Rainfall Overlay of Each Year by Month

##### Create Years of Interest List

In [ ]:
i = 1999
rf_years: list = []

while i < 2026:
	rf_years.append(i)
	i += 1

print(rf_years)

##### Dict of Dataframes for Each Year

In [ ]:
# List of years to add.
rf_dfs: dict = {}

# loop through all years in list and create dataframes.
for year in rf_years:
	df_name = f'df_rf_uk_{str(year)}'
	df_name = df_uk_rf_long[df_uk_rf_long['year'] == int(year)].copy()
	rf_dfs[str(year)] = df_name

	print(df_name.head(3))

##### Create Overlay Plot Month Rainfall for Each Year

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
for year in rf_years:
	df = rf_dfs[str(year)]
	df.loc[:, 'month_num'] = pd.to_datetime(df['month'], format='%b').dt.month
	plt.plot(df['month_num'], df['rainfall'], label= str(year))

# add figure elements
plt.title(f'UK Monthly Rainfall ({rf_years[0]} - {rf_years[-1]})')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm)')
plt.xticks(range(1,13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend(loc=0, ncols = 5)
plt.show()

This shows potential differences across the year for seasonal differences in rainfall. Depending on the sowing, growing and harvesting seasons of a crop, variations in more specific time periods could have a greater affect on yield than yearly averages.

#### 4.3 Rainfall Overlay of Each Year by Season

## [5. Combined Weather & Crop Data Analysis](#contents)

### [5.1 West Midlands Wheat Yield and Midlands Weather](#contents)

- Annual Wheat Yield for West Midlands
- Annual averages for each weather
- Correlation Heatmap

#### Data for West Midlands Wheat Yield and Annual Weather Averages

In [ ]:
query_wm_annual = """
	SELECT

		w."West Midlands" AS wm_wheat_yield,

		a.ann AS wm_airfrost,
		rd.ann AS wm_raindays,
		r.ann AS wm_rainfall,
		s.ann AS wm_sunshine,
		tx.ann AS wm_max_temp,
		tm.ann AS wm_mean_temp,
		tn.ann AS wm_min_temp
		

	FROM yields_wheat w

		JOIN Airfrost a ON w."Year" = a.year
		JOIN Rainfall r ON w."Year" = r.year
		JOIN Raindays1mm rd ON w."Year" = rd.year
		JOIN Sunshine s ON w."Year" = s.year
		JOIN Tmax tx ON w."Year" = tx.year
		JOIN Tmean tm ON w."Year" = tm.year
		JOIN Tmin tn ON w."Year" = tn.year
	
	WHERE
		a.area = 'Midlands'
		AND r.area = 'Midlands'
		AND rd.area = 'Midlands'
		AND s.area = 'Midlands'
		AND tx.area = 'Midlands'
		AND tm.area = 'Midlands'
		AND tn.area = 'Midlands'
		AND s.area = 'Midlands'
		AND w.year >= 1999 AND w.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_annual_wm_wheat = pd.read_sql(query_wm_annual, conn)

conn.close()

df_annual_wm_wheat.head()

#### West Midlands Wheat Yield Weather Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_annual_wm_wheat_heatmap.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('West Midlands Wheat Yield Weather Correlation Heatmap')
plt.show()

Observations:

- West Midlands Wheat Yield does not have any high correlations with any annual weather values.
- It may be that averaging over the year hides more granular weather factors, since wheat will only be influenced by the weather over the production cycle.
- The highest positive correlations are for annual monthly average maximum temperature and annual monthly average temperature.
- The highest negative correlation was for annual average monthly airfrost days.

### [5.2 West Midlands Monthly Maximum Temperature and Wheat Yields](#contents)

- Annual Wheat Yields for West Midlands
- Monthly Maximum Temperature
- Correlation Heatmap

##### West Midlands Monthly Max Temp and Wheat Yield Data

In [ ]:
query_wm_monthly_max_temp = """
	SELECT
		w."West Midlands" AS wm_wheat_yield,
		tx.jan,
		tx.feb,
		tx.mar,
		tx.apr,
		tx.may,
		tx.jun,
		tx.jul,
		tx.aug,
		tx.sep,
		tx.oct,
		tx.nov,
		tx.dec

	FROM yields_wheat w

		JOIN Tmax tx ON w."Year" = tx.year
	
	WHERE
		tx.area = 'Midlands'
		AND w.year >= 1999 AND w.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_wm_monthly_max_temp_wheat_yield = pd.read_sql(query_wm_monthly_max_temp, conn)

conn.close()

df_wm_monthly_max_temp_wheat_yield.head()

##### Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_wm_monthly_max_temp_wheat_yield.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('West Midlands Monthly Max Temp Wheat Yield Correlation Heatmap')
plt.show()

Observations:

- None of the months show a high correlation for Max Temperature with West Midlands Wheat Yield.
- The highest correlation is for the Month of February.

#### Scatter Plot

In [ ]:
# Remove unwanted columns
df_wm_march_max_temp_wheat_yield = df_wm_monthly_max_temp_wheat_yield.drop(columns=['jan','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'])
df_wm_march_max_temp_wheat_yield.head()

In [ ]:
# Define the arrays for the linear regression model.
x_uk_max_temp = np.array(df_wm_march_max_temp_wheat_yield['feb']).reshape(-1, 1)
y_wm_w_yield = np.array(df_wm_march_max_temp_wheat_yield['wm_wheat_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_uk_max_temp, y_wm_w_yield)

r_sq = model.score(x_uk_max_temp, y_wm_w_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_uk_max_temp)

# Plot the arrays and results of the model prediction
plt.scatter(x_uk_max_temp, y_wm_w_yield, color='blue', label='Actual Values')
plt.plot(x_uk_max_temp, y_pred, color='red', label='Model Fitted Line')
plt.title('West Midlands Wheat Yield vs February Maximum Daily Temperature')
plt.xlabel('Max Monthly Temperature')
plt.ylabel('WM Wheat Yield')
plt.legend()
plt.show()

Observation:

- There is a reasonable correlation coefficient (R = 0.070) between Maximum daily temperature and wheat yield in the West Midlands.

#### [5.2 West Midlands Monthly Airfrost Days and Wheat Yields](#contents)

- Annual Wheat Yields for West Midlands
- Monthly Airfrost Days
- Correlation Heatmap

##### West Midlands Monthly Airfrost Days and Wheat Yield Data

In [ ]:
query_wm_monthly_airfrost = """
	SELECT
		w."West Midlands" AS wm_wheat_yield,
		a.jan,
		a.feb,
		a.mar,
		a.apr,
		a.may,
		a.jun,
		a.jul,
		a.aug,
		a.sep,
		a.oct,
		a.nov,
		a.dec

	FROM yields_wheat w

		JOIN AirFrost a ON w."Year" = a.year
	
	WHERE
		a.area = 'Midlands'
		AND w.year >= 1999 AND w.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_wm_monthly_airfrost_wheat_yield = pd.read_sql(query_wm_monthly_airfrost, conn)

conn.close()

df_wm_monthly_airfrost_wheat_yield.head()

##### Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_wm_monthly_airfrost_wheat_yield.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('West Midlands Monthly Max Temp Wheat Yield Correlation Heatmap')
plt.show()

Observations:

- There are no especially strong monthly correlations for Airfrost days for Wheat Yield in the West Midlands.
- The most negatively correlated months are March (-0.35) and May (-0.34), with March marginally more impactful.

#### Scatter Plot

In [ ]:
df_wm_march_airfrost_wheat_yield = df_wm_monthly_airfrost_wheat_yield.drop(columns=['jan','feb','apr','may','jun','jul','aug','sep','oct','nov','dec'])
df_wm_march_airfrost_wheat_yield.head()

In [ ]:
# Define the arrays for the linear regression model.
x_uk_airfrost = np.array(df_wm_march_airfrost_wheat_yield['mar']).reshape(-1, 1)
y_wm_w_yield = np.array(df_wm_march_airfrost_wheat_yield['wm_wheat_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_uk_airfrost, y_wm_w_yield)

r_sq = model.score(x_uk_airfrost, y_wm_w_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_uk_airfrost)

# Plot the arrays and results of the model prediction
plt.scatter(x_uk_airfrost, y_wm_w_yield, color='blue', label='Actual Values')
plt.plot(x_uk_airfrost, y_pred, color='red', label='Model Fitted Line')
plt.title('West Midlands Wheat Yield vs Airfrost Days')
plt.xlabel('March Airfrost days')
plt.ylabel('WM Wheat Yield')
plt.legend()
plt.show()

Observation:

- The regression is a poor fit (0.12) between monthly airfrost days in March and Wheat Yield in the West Midlands.